# AI Career Path Advisor
## Multi-Agent Generative AI Project

This notebook implements a custom multi-agent career advisory system with:

- Specialized agents
- ReAct-style tool use
- Tool registry and tool calling
- Multi-agent orchestration
- Web and job-data integration hooks
- Loop detection
- Structured tracing
- Token and cost tracking
- Error handling and retry controls
- Career matching and skill-gap analysis
- Personalized career roadmap generation

The notebook is designed as a standalone project and does not import code from the instructor repository.


## 1. Install Dependencies

The project uses lightweight libraries that work in Google Colab.


In [1]:
!pip -q install openai ddgs pydantic requests

## 2. Imports and Configuration


In [2]:
import os
import re
import json
import time
import math
import hashlib
from dataclasses import dataclass, field
from typing import Any, Callable, Dict, List, Optional

from openai import OpenAI
from ddgs import DDGS
from pydantic import BaseModel, Field


import warnings

warnings.filterwarnings(
    "ignore",
    category=DeprecationWarning,
    module=r"jupyter_client\..*",
)


In [3]:
@dataclass
class AppConfig:
    model: str = "gemini-3.5-flash-lite"
    max_agent_steps: int = 8
    max_repeated_actions: int = 2
    temperature: float = 0.2
    max_search_results: int = 5

    # Gemini 3.5 Flash-Lite standard paid-tier pricing (USD per 1M tokens)
    input_cost_per_million: float = 0.30
    output_cost_per_million: float = 2.50


CONFIG = AppConfig()


## 3. API Keys

Add `GEMINI_API_KEY` to Google Colab Secrets.

An optional specialized jobs API can be connected later without changing the multi-agent architecture.


In [4]:
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError(
        "GEMINI_API_KEY was not found. Add it to Colab Secrets before continuing."
    )

client = OpenAI(
    api_key=GEMINI_API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

print("Gemini API connection configured successfully.")

Gemini API connection configured successfully.


## 4. Exceptions


In [5]:
class AgentError(Exception):
    pass

class ToolExecutionError(AgentError):
    pass

class MaxStepsError(AgentError):
    pass

class LoopDetectedError(AgentError):
    pass


## 5. Structured Observability

Every important agent and tool event is recorded. The trace includes execution time, token usage, estimated cost, tool calls, and errors.


In [6]:
@dataclass
class TraceEvent:
    timestamp: float
    event_type: str
    agent: str
    details: Dict[str, Any]

@dataclass
class RunMetrics:
    prompt_tokens: int = 0
    completion_tokens: int = 0
    total_tokens: int = 0
    estimated_cost: float = 0.0
    tool_calls: int = 0
    agent_calls: int = 0
    errors: int = 0
    started_at: float = field(default_factory=time.time)

    @property
    def elapsed_seconds(self) -> float:
        return time.time() - self.started_at

class Observer:
    def __init__(self):
        self.events: List[TraceEvent] = []
        self.metrics = RunMetrics()

    def log(self, event_type: str, agent: str, **details):
        self.events.append(
            TraceEvent(
                timestamp=time.time(),
                event_type=event_type,
                agent=agent,
                details=details,
            )
        )

    def record_usage(self, usage):
        if not usage:
            return
        prompt = getattr(usage, "prompt_tokens", 0) or 0
        completion = getattr(usage, "completion_tokens", 0) or 0
        total = getattr(usage, "total_tokens", prompt + completion) or 0

        self.metrics.prompt_tokens += prompt
        self.metrics.completion_tokens += completion
        self.metrics.total_tokens += total
        self.metrics.estimated_cost += (
            prompt / 1_000_000 * CONFIG.input_cost_per_million
            + completion / 1_000_000 * CONFIG.output_cost_per_million
        )

    def summary(self) -> Dict[str, Any]:
        return {
            "agent_calls": self.metrics.agent_calls,
            "tool_calls": self.metrics.tool_calls,
            "prompt_tokens": self.metrics.prompt_tokens,
            "completion_tokens": self.metrics.completion_tokens,
            "total_tokens": self.metrics.total_tokens,
            "estimated_cost_usd": round(self.metrics.estimated_cost, 6),
            "errors": self.metrics.errors,
            "elapsed_seconds": round(self.metrics.elapsed_seconds, 2),
        }

observer = Observer()


## 6. Loop Detection

Repeated tool actions are detected using normalized action signatures.


In [7]:
class LoopDetector:
    def __init__(
        self,
        max_repeats: int = 2,
        max_stagnant_steps: int = 3,
    ):
        self.max_repeats = max_repeats
        self.max_stagnant_steps = max_stagnant_steps
        self.history: Dict[str, int] = {}
        self.last_tool_name: Optional[str] = None
        self.stagnant_steps = 0

    def _signature(
        self,
        tool_name: str,
        arguments: Dict[str, Any],
    ) -> str:
        normalized = json.dumps(
            {
                "tool": tool_name,
                "arguments": arguments,
            },
            sort_keys=True,
            ensure_ascii=True,
        )

        return hashlib.sha256(
            normalized.encode()
        ).hexdigest()

    def check(
        self,
        tool_name: str,
        arguments: Dict[str, Any],
    ):
        signature = self._signature(
            tool_name,
            arguments,
        )

        self.history[signature] = (
            self.history.get(signature, 0) + 1
        )

        # Detect exact repetition
        if self.history[signature] > self.max_repeats:
            raise LoopDetectedError(
                f"Repeated action detected: "
                f"{tool_name} with identical arguments."
            )

        # Detect stagnation
        if tool_name == self.last_tool_name:
            self.stagnant_steps += 1
        else:
            self.stagnant_steps = 1
            self.last_tool_name = tool_name

        if self.stagnant_steps > self.max_stagnant_steps:
            raise LoopDetectedError(
                f"Stagnation detected: {tool_name} "
                f"was used repeatedly without changing strategy."
            )

## 7. Tool Registry


In [8]:
@dataclass
class Tool:
    name: str
    description: str
    parameters: Dict[str, Any]
    function: Callable[..., Any]

class ToolRegistry:
    def __init__(self):
        self._tools: Dict[str, Tool] = {}

    def register(
        self,
        name: str,
        description: str,
        parameters: Dict[str, Any],
        function: Callable[..., Any],
    ):
        self._tools[name] = Tool(
            name=name,
            description=description,
            parameters=parameters,
            function=function,
        )

    def schemas(self, allowed_tools: Optional[List[str]] = None) -> List[Dict[str, Any]]:
        names = allowed_tools or list(self._tools.keys())
        return [
            {
                "type": "function",
                "function": {
                    "name": self._tools[name].name,
                    "description": self._tools[name].description,
                    "parameters": self._tools[name].parameters,
                },
            }
            for name in names
            if name in self._tools
        ]

    def execute(self, name: str, arguments: Dict[str, Any]) -> Any:
        if name not in self._tools:
            raise ToolExecutionError(f"Unknown tool: {name}")

        try:
            return self._tools[name].function(**arguments)
        except Exception as exc:
            raise ToolExecutionError(f"Tool '{name}' failed: {exc}") from exc

registry = ToolRegistry()


## 8. Career Tools

These tools provide deterministic capabilities to the agents. They are separate from the language model.


In [9]:
def search_web(query: str, max_results: int = 5) -> List[Dict[str, str]]:
    max_results = max(1, min(int(max_results), 8))

    try:
        results = []

        with DDGS() as ddgs:
            search_results = ddgs.text(
                query,
                max_results=max_results,
            )

            for item in search_results:
                results.append(
                    {
                        "title": item.get("title", ""),
                        "url": item.get("href", ""),
                        "snippet": item.get("body", ""),
                    }
                )

        return results

    except Exception as exc:
        observer.log(
            "search_web_warning",
            "search_web",
            query=query,
            error=str(exc),
        )

        return []



registry.register(
    name="search_web",
    description=(
        "Search the public web for current career roles, skill requirements, "
        "technology trends, and labor-market information."
    ),
    parameters={
        "type": "object",
        "properties": {
            "query": {"type": "string"},
            "max_results": {
                "type": "integer",
                "minimum": 1,
                "maximum": 8,
                "default": 5,
            },
        },
        "required": ["query"],
    },
    function=search_web,
)


In [10]:
def calculate_match_score(
    user_skills: List[str],
    required_skills: List[str],
) -> Dict[str, Any]:
    normalize = lambda values: {
        re.sub(r"[^a-z0-9+#. ]", "", value.lower()).strip()
        for value in values
        if value and value.strip()
    }

    user = normalize(user_skills)
    required = normalize(required_skills)

    if not required:
        return {
            "score": 0.0,
            "matched_skills": [],
            "missing_skills": [],
        }

    matched = sorted(user.intersection(required))
    missing = sorted(required.difference(user))
    score = round((len(matched) / len(required)) * 100, 1)

    return {
        "score": score,
        "matched_skills": matched,
        "missing_skills": missing,
    }

registry.register(
    name="calculate_match_score",
    description="Calculate a deterministic skill-match score for a career role.",
    parameters={
        "type": "object",
        "properties": {
            "user_skills": {
                "type": "array",
                "items": {"type": "string"},
            },
            "required_skills": {
                "type": "array",
                "items": {"type": "string"},
            },
        },
        "required": ["user_skills", "required_skills"],
    },
    function=calculate_match_score,
)


In [11]:
def prioritize_skill_gaps(
    missing_skills: List[str],
    high_priority_skills: List[str],
) -> Dict[str, List[str]]:
    missing = [skill.strip() for skill in missing_skills if skill.strip()]
    priority_set = {skill.lower().strip() for skill in high_priority_skills}

    high = [skill for skill in missing if skill.lower() in priority_set]
    standard = [skill for skill in missing if skill.lower() not in priority_set]

    return {
        "high_priority": high,
        "standard_priority": standard,
    }

registry.register(
    name="prioritize_skill_gaps",
    description="Prioritize missing career skills using a supplied high-priority skill list.",
    parameters={
        "type": "object",
        "properties": {
            "missing_skills": {
                "type": "array",
                "items": {"type": "string"},
            },
            "high_priority_skills": {
                "type": "array",
                "items": {"type": "string"},
            },
        },
        "required": ["missing_skills", "high_priority_skills"],
    },
    function=prioritize_skill_gaps,
)


## 9. Jooble Jobs API Integration

This integration connects the system to current Saudi Arabia job listings through the Jooble API while keeping the external service isolated from the agent architecture.

In [12]:
import requests

try:
    from google.colab import userdata
    JOOBLE_API_KEY = userdata.get("JOOBLE_API_KEY")
except Exception:
    JOOBLE_API_KEY = os.getenv("JOOBLE_API_KEY")

if not JOOBLE_API_KEY:
    raise ValueError(
        "JOOBLE_API_KEY was not found. Add it to Colab Secrets before continuing."
    )

latest_job_results = []


def search_jobs_external(
    query: str,
    location: str,
    limit: int = 10,
) -> Dict[str, Any]:
    global latest_job_results

    url = f"https://sa.jooble.org/api/{JOOBLE_API_KEY}"

    payload = {
        "keywords": query,
        "location": location,
        "page": 1,
    }

    response = requests.post(
        url,
        json=payload,
        timeout=20,
    )
    response.raise_for_status()

    data = response.json()
    jobs = data.get("jobs", [])[:limit]

    cleaned_jobs = []

    for job in jobs:
        cleaned_jobs.append(
            {
                "title": job.get("title", ""),
                "company": (
    job.get("company", "").strip()
    if len(job.get("company", "").strip()) > 1
    else "Not specified"
),
                "location": job.get("location", ""),
                "snippet": job.get("snippet", ""),
                "salary": job.get("salary", ""),
                "source": job.get("source", ""),
                "link": job.get("link", ""),
                "updated": job.get("updated", ""),
            }
        )

    latest_job_results = cleaned_jobs.copy()

    return {
        "status": "success",
        "query": query,
        "location": location,
        "total_available": data.get("totalCount", 0),
        "returned_jobs": len(cleaned_jobs),
        "jobs": cleaned_jobs,
    }


registry.register(
    name="search_jobs_external",
    description=(
        "Search current job vacancies using the Jooble Saudi Arabia Jobs API. "
        "Use this tool when real job-market evidence is needed for a career recommendation."
    ),
    parameters={
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Job title or career keywords to search for.",
            },
            "location": {
                "type": "string",
                "description": "Target job location, such as Riyadh or Saudi Arabia.",
            },
            "limit": {
                "type": "integer",
                "minimum": 1,
                "maximum": 25,
                "default": 10,
            },
        },
        "required": ["query", "location"],
    },
    function=search_jobs_external,
)

## BOUNS: RAG Career Knowledge Base

This section implements a lightweight Retrieval-Augmented Generation (RAG) layer that retrieves relevant career knowledge and supplies it as grounded context to the multi-agent workflow.

In [13]:
!pip -q install pypdf sentence-transformers

In [14]:
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import numpy as np

PDF_PATH = "/content/IP_Career_Listings.pdf"

reader = PdfReader(PDF_PATH)

pdf_pages = []
for page_number, page in enumerate(reader.pages, start=1):
    text = page.extract_text()
    if text and text.strip():
        pdf_pages.append({
            "page": page_number,
            "text": text.strip()
        })

print(f"PDF loaded successfully.")
print(f"Pages extracted: {len(pdf_pages)}")
print(f"First page preview:\n{pdf_pages[0]['text'][:500]}")

PDF loaded successfully.
Pages extracted: 12
First page preview:
National Center for O*NET Development v1.4                                                                                                                                                   Page 1 of 12 
O*NET CAREER LISTINGS 
 
O*NET Careers by Interest and Job Zone 
**The career was assigned to the group based on its second highest interest. 
                     ***The career was assigned to the group based on its third highest interest. 
 
REALISTIC — JOB ZONE 1 (Little or No Preparation Need


In [15]:
def create_chunks(pages, chunk_size=800, overlap=150):
    chunks = []

    for page in pages:
        text = page["text"]
        start = 0

        while start < len(text):
            end = start + chunk_size
            chunk_text = text[start:end].strip()

            if chunk_text:
                chunks.append({
                    "page": page["page"],
                    "text": chunk_text
                })

            start += chunk_size - overlap

    return chunks


career_chunks = create_chunks(pdf_pages)

print(f"Total chunks created: {len(career_chunks)}")
print(f"\nExample chunk:\n{career_chunks[0]['text'][:500]}")

Total chunks created: 43

Example chunk:
National Center for O*NET Development v1.4                                                                                                                                                   Page 1 of 12 
O*NET CAREER LISTINGS 
 
O*NET Careers by Interest and Job Zone 
**The career was assigned to the group based on its second highest interest. 
                     ***The career was assigned to the group based on its third highest interest. 
 
REALISTIC — JOB ZONE 1 (Little or No Preparation Need


In [16]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_texts = [chunk["text"] for chunk in career_chunks]

chunk_embeddings = embedding_model.encode(
    chunk_texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Embeddings created successfully.")
print(f"Number of embeddings: {len(chunk_embeddings)}")
print(f"Embedding dimension: {chunk_embeddings.shape[1]}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Embeddings created successfully.
Number of embeddings: 43
Embedding dimension: 384


In [17]:
def retrieve_pdf_knowledge(query, top_k=3):
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )[0]

    similarity_scores = np.dot(chunk_embeddings, query_embedding)

    top_indices = np.argsort(similarity_scores)[::-1][:top_k]

    results = []

    for index in top_indices:
        results.append({
            "page": career_chunks[index]["page"],
            "text": career_chunks[index]["text"],
            "score": float(similarity_scores[index])
        })

    return results


test_results = retrieve_pdf_knowledge(
    "I am interested in software development, artificial intelligence, and data science."
)

for result in test_results:
    print(f"\nPage: {result['page']}")
    print(f"Similarity: {result['score']:.3f}")
    print(result["text"][:500])
    print("-" * 60)


Page: 12
Similarity: 0.382
diators, & Conciliators** Lawyers** 
Archivists Librarians & Media Collections Specialists 
Biostatisticians** Mathematicians** 
Computer & Information Research Scientists** Operations Research Analysts** 
Curators Radiologists** 
Economists** Statisticians** 
Financial Quantitative Analysts** Survey Researchers** 
Geneticists** Teaching Assistants, Postsecondary** 
 
 
 
 
 
Sponsored by the U.S. Department of Labor, Employment & Training Administration & developed by the 
National Center for O
------------------------------------------------------------

Page: 4
Similarity: 0.370
Economists Operations Research Analysts 
Environmental Restoration Planners Optometrists 
Financial Quantitative Analysts Orthodontists 
Genetic Counselors Physicists 
Human Factors Engineers & Ergonomists Political Scientists 
Industrial-Organizational Psychologists Soil & Plant Scientists 
Industrial Ecologists Statisticians 
  
INVESTIGATIVE | Job Zones 4 - 5
------------------

In [18]:
def retrieve_career_knowledge(
    query: str,
    top_k: int = 3,
):
    """
    Retrieve the most relevant career knowledge from the O*NET PDF
    using semantic similarity.
    """

    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )[0]

    similarity_scores = np.dot(
        chunk_embeddings,
        query_embedding
    )

    top_indices = np.argsort(
        similarity_scores
    )[::-1][:top_k]

    retrieved_documents = []

    for index in top_indices:
        retrieved_documents.append({
            "source": "O*NET Career Listings",
            "page": career_chunks[index]["page"],
            "content": career_chunks[index]["text"],
            "retrieval_score": float(
                similarity_scores[index]
            ),
        })

    return retrieved_documents


print("PDF-based semantic RAG retriever configured successfully.")

PDF-based semantic RAG retriever configured successfully.


In [19]:
rag_test = retrieve_career_knowledge(
    "Computer Science graduate interested in AI, machine learning, and software development."
)

for item in rag_test:
    print(f"Source: {item['source']}")
    print(f"Page: {item['page']}")
    print(f"Score: {item['retrieval_score']:.3f}")
    print(item["content"][:300])
    print("-" * 60)

Source: O*NET Career Listings
Page: 12
Score: 0.405
diators, & Conciliators** Lawyers** 
Archivists Librarians & Media Collections Specialists 
Biostatisticians** Mathematicians** 
Computer & Information Research Scientists** Operations Research Analysts** 
Curators Radiologists** 
Economists** Statisticians** 
Financial Quantitative Analysts** Surve
------------------------------------------------------------
Source: O*NET Career Listings
Page: 12
Score: 0.389
Analysts Management Analysts 
Clinical Data Managers Online Merchants 
Compensation, Benefits, & Job Analysis Specialists Regulatory Affairs Specialists 
Computer & Information Systems Managers Security Management Specialists 
Database Administrators Surveyors 
Environmental Compliance Inspectors Te
------------------------------------------------------------
Source: O*NET Career Listings
Page: 2
Score: 0.378
ion Needed) 
O*NET Career Title O*NET Career Title 
Airline Pilots, Copilots, & Flight Engineers Geological Technicians 


## 10. Base ReAct Agent

The base agent repeatedly decides whether to use a tool or return a final answer. Tool results are added back to the conversation before the next reasoning step.


In [20]:
class BaseAgent:
    def __init__(
        self,
        name: str,
        system_prompt: str,
        allowed_tools: Optional[List[str]] = None,
        max_steps: Optional[int] = None,
        max_tool_calls: int = 4,
    ):
        self.name = name
        self.system_prompt = system_prompt
        self.allowed_tools = allowed_tools or []
        self.max_steps = max_steps or CONFIG.max_agent_steps
        self.max_tool_calls = max_tool_calls

    def run(self, task: str) -> Dict[str, Any]:
        observer.metrics.agent_calls += 1
        observer.log("agent_start", self.name, task=task)

        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": task},
        ]

        detector = LoopDetector(CONFIG.max_repeated_actions)
        tool_schemas = registry.schemas(self.allowed_tools)

        tool_calls_used = 0
        budget_notice_added = False

        for step in range(1, self.max_steps + 1):
            observer.log(
                "agent_step",
                self.name,
                step=step,
                tool_calls_used=tool_calls_used,
            )

            try:
                tools_available = (
                    bool(tool_schemas)
                    and tool_calls_used < self.max_tool_calls
                )

                if (
                    tool_schemas
                    and not tools_available
                    and not budget_notice_added
                ):
                    messages.append(
                        {
                            "role": "system",
                            "content": (
                                "The tool-call budget is exhausted. "
                                "Do not request more tools. "
                                "Return the final answer using the "
                                "evidence already collected."
                            ),
                        }
                    )
                    budget_notice_added = True

                kwargs = {
                    "model": CONFIG.model,
                    "messages": messages,
                    "temperature": CONFIG.temperature,
                }

                if tools_available:
                    kwargs["tools"] = tool_schemas
                    kwargs["tool_choice"] = "auto"

                response = None

                # Retry temporary API errors and rate limits.
                for attempt in range(5):
                    try:
                        response = client.chat.completions.create(**kwargs)
                        break

                    except Exception as exc:
                        error_text = str(exc)

                        # Rate limit: wait for the next quota window.
                        if "429" in error_text and attempt < 4:
                            wait_seconds = 30

                            observer.log(
                                "rate_limit_retry",
                                self.name,
                                attempt=attempt + 1,
                                wait_seconds=wait_seconds,
                                error="429 Rate Limit",
                            )

                            print(
                                f"{self.name}: rate limit reached. "
                                f"Waiting {wait_seconds} seconds..."
                            )

                            time.sleep(wait_seconds)
                            continue

                        # Temporary Gemini service overload.
                        if "503" in error_text and attempt < 4:
                            wait_seconds = 10 * (attempt + 1)

                            observer.log(
                                "service_retry",
                                self.name,
                                attempt=attempt + 1,
                                wait_seconds=wait_seconds,
                                error="503 Service Unavailable",
                            )

                            print(
                                f"{self.name}: Gemini is busy. "
                                f"Retrying in {wait_seconds} seconds..."
                            )

                            time.sleep(wait_seconds)
                            continue

                        raise

                if response is None:
                    raise AgentError(
                        f"{self.name} did not receive an LLM response."
                    )

                observer.record_usage(response.usage)

                message = response.choices[0].message

                if not message.tool_calls:
                    answer = (message.content or "").strip()

                    if not answer:
                        raise AgentError(
                            f"{self.name} returned an empty final answer."
                        )

                    observer.log(
                        "agent_finish",
                        self.name,
                        step=step,
                        tool_calls_used=tool_calls_used,
                        answer_preview=answer[:300],
                    )

                    return {
                        "answer": answer,
                        "steps": step,
                        "tool_calls_used": tool_calls_used,
                        "status": "completed",
                    }

                for tool_call in message.tool_calls:
                    tool_name = tool_call.function.name

                    if tool_name not in self.allowed_tools:
                        raise ToolExecutionError(
                            f"{self.name} attempted to use "
                            f"unauthorized tool: {tool_name}"
                        )

                messages.append(message)

                for tool_call in message.tool_calls:
                    tool_name = tool_call.function.name

                    if tool_calls_used >= self.max_tool_calls:
                        observer.log(
                            "tool_budget_exhausted",
                            self.name,
                            tool=tool_name,
                            max_tool_calls=self.max_tool_calls,
                        )

                        messages.append(
                            {
                                "role": "tool",
                                "tool_call_id": tool_call.id,
                                "content": json.dumps(
                                    {
                                        "status": "skipped",
                                        "reason": (
                                            "Tool-call budget exhausted. "
                                            "Use the evidence already collected "
                                            "and return the final answer."
                                        ),
                                    }
                                ),
                            }
                        )
                        continue

                    try:
                        arguments = json.loads(
                            tool_call.function.arguments or "{}"
                        )
                    except json.JSONDecodeError as exc:
                        raise ToolExecutionError(
                            f"Invalid tool arguments from "
                            f"{self.name}: {exc}"
                        )

                    detector.check(tool_name, arguments)

                    tool_calls_used += 1
                    observer.metrics.tool_calls += 1

                    observer.log(
                        "tool_start",
                        self.name,
                        tool=tool_name,
                        arguments=arguments,
                        tool_call_number=tool_calls_used,
                    )

                    started = time.time()

                    try:
                        result = registry.execute(
                            tool_name,
                            arguments,
                        )
                    except Exception as exc:
                        raise ToolExecutionError(
                            f"Tool '{tool_name}' failed for "
                            f"{self.name}: {exc}"
                        ) from exc

                    elapsed = time.time() - started

                    observer.log(
                        "tool_finish",
                        self.name,
                        tool=tool_name,
                        elapsed_seconds=round(elapsed, 3),
                        result_preview=str(result)[:500],
                    )

                    messages.append(
                        {
                            "role": "tool",
                            "tool_call_id": tool_call.id,
                            "content": json.dumps(
                                result,
                                ensure_ascii=False,
                                default=str,
                            ),
                        }
                    )

            except (
                LoopDetectedError,
                ToolExecutionError,
                AgentError,
            ):
                observer.metrics.errors += 1
                raise

            except Exception as exc:
                observer.metrics.errors += 1

                observer.log(
                    "agent_error",
                    self.name,
                    step=step,
                    error=repr(exc),
                )

                raise AgentError(
                    f"{self.name} failed at step {step}: {exc}"
                ) from exc

        observer.metrics.errors += 1

        observer.log(
            "max_steps_reached",
            self.name,
            max_steps=self.max_steps,
            tool_calls_used=tool_calls_used,
        )

        raise MaxStepsError(
            f"{self.name} exceeded the maximum of "
            f"{self.max_steps} steps."
        )

## 11. Specialized Agent Prompts


In [21]:
PROFILE_PROMPT = '''
You are the Profile Analysis Agent in an AI Career Path Advisor.

Your responsibility is to convert the user's background into a concise professional profile.

Identify:
- education and field
- current technical and transferable skills
- interests
- career preferences
- location constraints
- experience level
- important missing information

Do not recommend a final career path.
Do not invent qualifications that the user did not provide.
Return a structured profile that downstream agents can use.
'''

CAREER_RESEARCH_PROMPT = '''
You are the Career Research Agent in an AI Career Path Advisor.

Your responsibility is to identify realistic career paths that fit the supplied profile.

Use tools selectively and efficiently:
- Use web search only when current market information, role requirements, or industry trends are needed.
- Use the Jooble jobs search tool when current Saudi job-market evidence is useful.
- Prefer one focused web search and one focused jobs search when possible.
- Do not repeat equivalent searches.
- Stop using tools as soon as enough evidence has been collected.
- After collecting sufficient evidence, return the final research result immediately.

For each strong candidate career:
- provide the role name
- summarize why it fits
- identify commonly requested skills
- distinguish evidence from inference
- retain useful source URLs when tools provide them

Prefer a small set of well-supported career paths over a long generic list.
Do not make the final recommendation.
Do not invent market evidence, job listings, requirements, or sources.
'''

SKILL_GAP_PROMPT = '''
You are the Skill Gap Analysis Agent in an AI Career Path Advisor.

Compare the user's current skills with the requirements of the candidate career paths.

Use the deterministic match-score tool when sufficient skill lists are available.
Use the skill-priority tool when useful.

For each career path:
- identify matched skills
- identify missing skills
- estimate readiness
- explain the most important gaps

Do not invent skills for the user.
Do not make the final career decision.
'''

ROADMAP_PROMPT = '''
You are the Career Roadmap Agent in an AI Career Path Advisor.

Create a practical learning and development roadmap for the strongest career candidates.
Base the roadmap only on the supplied profile, research, and skill-gap analysis.

The roadmap should:
- prioritize high-impact gaps
- use realistic phases
- include portfolio or practice milestones
- avoid unnecessary learning
- explain what success looks like at each phase

Do not claim guaranteed employment outcomes.
'''

REVIEWER_PROMPT = '''
You are the Critical Review Agent in an AI Career Path Advisor.

Audit the proposed career analysis before the final report.

Check for:
- unsupported claims
- contradictions
- weak career-role justification
- recommendations that ignore the user's constraints
- missing evidence
- unrealistic roadmap steps

Return:
1. PASS or REVISE
2. concise reasons
3. specific corrections if revision is needed
'''

WRITER_PROMPT = '''
You are the Final Career Report Agent.

Synthesize the validated outputs from all previous agents into one professional report.

The report must include:
- User Profile Summary
- Recommended Career Paths
- Best-Fit Career and rationale
- Match evidence
- Strengths
- Skill Gaps
- Development Roadmap
- Current Job Opportunities, when Jooble job data is available
- Sources used when available
- Limitations

For Current Job Opportunities:
- Select only jobs that are relevant to the user's profile, recommended career paths, and experience level.
- Use only the job data supplied from Jooble.
- Do not invent or modify job titles, company names, locations, sources, or application links.
- Exclude clearly unrelated or overly senior positions.
- For each selected opportunity, include:
  - Job Title
  - Company
  - Location
  - Source
  - Apply Link
- If Jooble data is available but no suitable opportunity matches the user, clearly state that no suitable current opportunities were found.

Clearly separate observed evidence from recommendations.
Do not invent facts, scores, sources, job opportunities, or user qualifications.
'''


## 12. Specialized Agents


In [22]:
profile_agent = BaseAgent(
    name="ProfileAgent",
    system_prompt=PROFILE_PROMPT,
    max_steps=2,
)

career_research_agent = BaseAgent(
    name="CareerResearchAgent",
    system_prompt=CAREER_RESEARCH_PROMPT,
    allowed_tools=["search_web", "search_jobs_external"],
    max_steps=4,
    max_tool_calls=2,
)

skill_gap_agent = BaseAgent(
    name="SkillGapAgent",
    system_prompt=SKILL_GAP_PROMPT,
    allowed_tools=["calculate_match_score", "prioritize_skill_gaps"],
    max_steps=4,
    max_tool_calls=2,
)

roadmap_agent = BaseAgent(
    name="RoadmapAgent",
    system_prompt=ROADMAP_PROMPT,
    max_steps=2,
)

reviewer_agent = BaseAgent(
    name="ReviewerAgent",
    system_prompt=REVIEWER_PROMPT,
    max_steps=2,
)

writer_agent = BaseAgent(
    name="WriterAgent",
    system_prompt=WRITER_PROMPT,
    max_steps=2,
)

## 13. Multi-Agent Orchestrator

The orchestrator implements a sequential pipeline with a quality-review stage and one controlled revision path.


In [23]:
class CareerAdvisorOrchestrator:
    def __init__(self):
        self.max_revision_rounds = 1

    @staticmethod
    def _pack(title: str, value: Any) -> str:
        return f"\n\n===== {title} =====\n{value}"

    @staticmethod
    def _get_review_decision(review: str) -> str:
        match = re.search(r"\b(PASS|REVISE)\b", review.upper())

        if match:
            return match.group(1)

        return "PASS"

    def run(self, user_request: str) -> Dict[str, Any]:
        observer.log(
            "orchestrator_start",
            "Orchestrator",
            request=user_request,
        )

        # Clear job results from any previous run
        latest_job_results.clear()

        # Analyze the user profile
        profile = profile_agent.run(user_request)["answer"]

        # Retrieve relevant career knowledge from the PDF RAG layer
        rag_results = retrieve_career_knowledge(
            user_request,
            top_k=3,
        )

        # Format retrieved PDF knowledge for the agents
        rag_context = "\n\n".join(
            [
                f"Source: {item['source']}\n"
                f"Page: {item['page']}\n"
                f"Knowledge: {item['content']}\n"
                f"Retrieval Score: {item['retrieval_score']:.3f}"
                for item in rag_results
            ]
        )

        observer.log(
            "rag_retrieval",
            "Orchestrator",
            retrieved_documents=len(rag_results),
            sources=[
                f"{item['source']} - Page {item['page']}"
                for item in rag_results
            ],
        )

        research_task = (
            "Analyze the following user profile and research suitable career paths. "
            "Use the retrieved career knowledge as additional grounded context."
            + self._pack("USER PROFILE", profile)
            + self._pack("RETRIEVED CAREER KNOWLEDGE", rag_context)
        )

        research = career_research_agent.run(research_task)["answer"]

        gap_task = (
            "Perform a skill-gap analysis using the profile and career research."
            + self._pack("USER PROFILE", profile)
            + self._pack("CAREER RESEARCH", research)
        )

        gaps = skill_gap_agent.run(gap_task)["answer"]

        roadmap_task = (
            "Create a development roadmap using the following evidence."
            + self._pack("USER PROFILE", profile)
            + self._pack("CAREER RESEARCH", research)
            + self._pack("SKILL GAP ANALYSIS", gaps)
        )

        roadmap = roadmap_agent.run(roadmap_task)["answer"]

        review_task = (
            "Review the complete analysis for quality and evidence."
            + self._pack("USER PROFILE", profile)
            + self._pack("CAREER RESEARCH", research)
            + self._pack("SKILL GAP ANALYSIS", gaps)
            + self._pack("ROADMAP", roadmap)
        )

        review = reviewer_agent.run(review_task)["answer"]

        review_decision = self._get_review_decision(review)

        observer.log(
            "review_decision",
            "Orchestrator",
            decision=review_decision,
        )

        if review_decision == "REVISE":
            revision_task = (
                "Revise the roadmap using the review feedback. "
                "Preserve supported content and correct only identified weaknesses."
                + self._pack("USER PROFILE", profile)
                + self._pack("CAREER RESEARCH", research)
                + self._pack("SKILL GAP ANALYSIS", gaps)
                + self._pack("CURRENT ROADMAP", roadmap)
                + self._pack("REVIEW FEEDBACK", review)
            )

            roadmap = roadmap_agent.run(revision_task)["answer"]

        job_opportunities = json.dumps(
            latest_job_results,
            ensure_ascii=False,
            indent=2,
        )

        final_task = (
            "Create the final career advisory report from the validated material."
            + self._pack("ORIGINAL USER REQUEST", user_request)
            + self._pack("USER PROFILE", profile)
            + self._pack("CAREER RESEARCH", research)
            + self._pack("SKILL GAP ANALYSIS", gaps)
            + self._pack("ROADMAP", roadmap)
            + self._pack("QUALITY REVIEW", review)
            + self._pack("RETRIEVED CAREER KNOWLEDGE", rag_context)
            + self._pack(
                "CURRENT JOB OPPORTUNITIES FROM JOOBLE",
                job_opportunities,
            )
            + """

IMPORTANT JOB OPPORTUNITY INSTRUCTIONS:
- Add a section titled "Current Job Opportunities".
- Select only the most relevant jobs for the user's profile and experience level.
- Use only jobs provided in CURRENT JOB OPPORTUNITIES FROM JOOBLE.
- Do not invent job titles, companies, locations, or links.
- For each selected job, show:
  Job Title
  Company
  Location
  Source
  Apply Link
- Exclude clearly unsuitable senior or unrelated positions.
- If no suitable jobs are available, state that no suitable current opportunities were found.
"""
        )

        final_report = writer_agent.run(final_task)["answer"]

        observer.log(
            "orchestrator_finish",
            "Orchestrator",
            review_decision=review_decision,
        )

        return {
            "answer": final_report,
            "metadata": {
                "profile": profile,
                "rag_results": rag_results,
                "research": research,
                "skill_gap_analysis": gaps,
                "roadmap": roadmap,
                "review": review,
                "review_decision": review_decision,
                "job_opportunities": latest_job_results.copy(),
                "metrics": observer.summary(),
            },
        }


career_advisor = CareerAdvisorOrchestrator()

## 14. Run the AI Career Path Advisor

Edit the example profile or replace it with another user scenario.


In [24]:
user_request = """
I am a recent Marketing graduate based in Saudi Arabia.

My current skills include:
- Communication
- Social Media Management
- Content Creation
- Market Research

My interests include:
- Digital Marketing
- Branding
- Advertising
- Customer Behavior

I am looking for an entry-level career path in Saudi Arabia.
I want to know which career paths best match my current profile,
which skills I am missing, and what I should learn next.
"""

result = career_advisor.run(user_request)

print("=== FINAL CAREER ADVISORY REPORT ===")
print(result["answer"])

=== FINAL CAREER ADVISORY REPORT ===
# Final Career Advisory Report

---

## User Profile Summary
* **Education:** Recent Graduate in Marketing
* **Experience Level:** Entry-level / Graduate
* **Location & Constraints:** Based in Saudi Arabia; seeking local opportunities.
* **Current Skills:** Communication, Social Media Management, Content Creation, Market Research.
* **Interests:** Digital Marketing, Branding, Advertising, Customer Behavior.
* **Career Preferences:** Entry-level career path within Saudi Arabia.

---

## Recommended Career Paths
Based on your profile and local market indicators in Saudi Arabia, three viable entry-level career paths have been identified:
1. **Digital Marketing Executive / Specialist** *(Focus on data-driven campaigns, digital platforms, and multi-channel marketing)*
2. **Social Media & Content Specialist** *(Focus on brand voice, content creation, community engagement, and regional trend alignment)*
3. **Market Research Analyst / Junior Insights Associ

## 15. Observability Summary


In [25]:
metrics = result["metadata"]["metrics"]

print("=== OBSERVABILITY SUMMARY ===")
print(f"Agent Calls       : {metrics['agent_calls']}")
print(f"Tool Calls        : {metrics['tool_calls']}")
print(f"Prompt Tokens     : {metrics['prompt_tokens']}")
print(f"Completion Tokens : {metrics['completion_tokens']}")
print(f"Total Tokens      : {metrics['total_tokens']}")
print(f"Estimated Cost    : ${metrics['estimated_cost_usd']:.6f}")
print(f"Errors            : {metrics['errors']}")
print(f"Elapsed Time      : {metrics['elapsed_seconds']} seconds")

=== OBSERVABILITY SUMMARY ===
Agent Calls       : 6
Tool Calls        : 3
Prompt Tokens     : 20959
Completion Tokens : 3860
Total Tokens      : 24819
Estimated Cost    : $0.015938
Errors            : 0
Elapsed Time      : 170.9 seconds


## 16. Execution Trace


In [26]:
print("=== EXECUTION TRACE ===")

if not observer.events:
    print("No trace events were recorded.")
else:
    for index, event in enumerate(observer.events, start=1):
        elapsed = event.timestamp - observer.metrics.started_at

        print(
            f"{index:03d} | "
            f"+{elapsed:7.2f}s | "
            f"{event.event_type:<22} | "
            f"{event.agent:<22} | "
            f"{event.details}"
        )

=== EXECUTION TRACE ===
001 | +  55.56s | orchestrator_start     | Orchestrator           | {'request': '\nI am a recent Marketing graduate based in Saudi Arabia.\n\nMy current skills include:\n- Communication\n- Social Media Management\n- Content Creation\n- Market Research\n\nMy interests include:\n- Digital Marketing\n- Branding\n- Advertising\n- Customer Behavior\n\nI am looking for an entry-level career path in Saudi Arabia.\nI want to know which career paths best match my current profile,\nwhich skills I am missing, and what I should learn next.\n'}
002 | +  55.56s | agent_start            | ProfileAgent           | {'task': '\nI am a recent Marketing graduate based in Saudi Arabia.\n\nMy current skills include:\n- Communication\n- Social Media Management\n- Content Creation\n- Market Research\n\nMy interests include:\n- Digital Marketing\n- Branding\n- Advertising\n- Customer Behavior\n\nI am looking for an entry-level career path in Saudi Arabia.\nI want to know which career pa

## 17. Component Tests

These tests verify deterministic project components without consuming additional LLM calls.


In [27]:
def run_component_tests():
    test_registry = registry.schemas()
    assert any(
        item["function"]["name"] == "calculate_match_score"
        for item in test_registry
    )

    score = calculate_match_score(
        ["Python", "SQL", "Machine Learning"],
        ["Python", "SQL", "Docker", "Machine Learning"],
    )
    assert score["score"] == 75.0
    assert "docker" in score["missing_skills"]

    detector = LoopDetector(max_repeats=1)
    detector.check("search_web", {"query": "AI Engineer"})

    try:
        detector.check("search_web", {"query": "AI Engineer"})
        raise AssertionError("Loop detector did not raise an error.")
    except LoopDetectedError:
        pass

    priorities = prioritize_skill_gaps(
        ["Docker", "Cloud", "Communication"],
        ["Docker", "Cloud"],
    )
    assert set(priorities["high_priority"]) == {"Docker", "Cloud"}

    # Test PDF-based semantic RAG retrieval
    rag_results = retrieve_career_knowledge(
        "software development artificial intelligence data science"
    )

    assert len(rag_results) > 0

    assert all(
        item["source"] == "O*NET Career Listings"
        for item in rag_results
    )

    assert all(
        "page" in item
        and "content" in item
        and "retrieval_score" in item
        for item in rag_results
    )

    assert all(
        isinstance(item["retrieval_score"], float)
        for item in rag_results
    )

    print("All component tests passed.")


run_component_tests()

All component tests passed.


## 18. Evaluation Coverage

### Agent Architecture
- Multiple specialized agents with clearly defined responsibilities
- ReAct-style iterative reasoning and tool use
- Structured tool registry with function schemas
- Multi-agent orchestration through `CareerAdvisorOrchestrator`
- Reviewer-driven quality control and controlled revision path
- Deterministic career-analysis tools for match scoring and skill-gap prioritization
- Current web research capability using DDGS
- Real-world job-market integration using the Jooble Jobs API
- Explicit tool authorization for each agent

### Observability and Reliability
- Structured execution tracing across agents and tools
- Agent start, step, finish, and error event logging
- Tool start and finish event logging
- Prompt, completion, and total token tracking
- Estimated cost tracking
- Agent-call and tool-call counting
- Runtime measurement
- Repeated-action loop detection
- Maximum-step protection
- Per-agent tool-call budget
- Tool-budget exhaustion handling
- Explicit custom exceptions and centralized error tracking
- Graceful handling of empty web-search results

### Engineering Quality
- Clear separation of agent, tool, orchestration, and observability responsibilities
- Reusable agent and utility classes
- Centralized configuration object
- Deterministic component tests
- Isolated external service integrations
- API credentials loaded securely from Colab Secrets or environment variables
- No hard-coded API secrets

### RAG Extension
A PDF-based semantic RAG system was integrated into the workflow using O*NET Career Listings.
The PDF is extracted, divided into text chunks, and converted into embeddings using SentenceTransformers.
Semantic similarity is used to retrieve the most relevant career knowledge for each user query.
The retrieved knowledge provides grounded context to the Career Research Agent and Final Report Agent.
PDF-based RAG retrieval is also covered by component tests.

## 19. Final Submission Checklist

Before final submission:

1. Confirm that the Gemini model used for the final demonstration is available and supports the required agent workflow.
2. Verify the Jooble Jobs API integration with a live Saudi Arabia job search.
3. Verify DDGS web search and graceful handling of empty search results.
4. Confirm that all API credentials are loaded securely from Colab Secrets and are not hard-coded.
5. Run the complete multi-agent workflow with a representative user profile.
6. Verify that each agent performs its assigned responsibility correctly.
7. Confirm that the CareerResearchAgent respects its tool-call budget.
8. Verify reviewer decision handling and the controlled revision path.
9. Review the observability summary for agent calls, tool calls, tokens, errors, cost, and runtime.
10. Review the execution trace for agent steps, tool usage, review decisions, and failures.
11. Run all deterministic component tests and confirm that they pass.
12. Save representative successful outputs and traces for the final demonstration.
13. Document the architecture, agents, tools, external integrations, setup steps, limitations, and execution instructions.
14. Confirm that the integrated RAG layer retrieves relevant career knowledge and that its component tests pass.